In [ ]:
%%html
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>

# Outline
0. Part 0 — Intro/Setup
1. Part 1 — OpenAI Python APIs
2. **Part 2 — OpenAI Agents SDK**
   * 02-00. Agents SDK Introduction: building blocks, ReAct pattern
   * 02-01. Single Agent System — Python Tutor: `Agent`, `Runner.run()`, `RunResult`
   * 02-02. Conversation State in Agents: `previous_response_id`, `SQLiteSession`, `conversation_id`, `result.to_input_list()`
   * 02-03. Streaming Text and Events: `Runner.run_streamed()`, `StreamEvent`, `run_item_stream_event`
   * 02-04. Python Tutor with a Model-Backed Input Guardrail: `@input_guardrail`, `GuardrailFunctionOutput`, `InputGuardrailTripwireTriggered`
   * **02-05. Tools**
      * 02-05-00. Tools Overview: hosted, local, and custom tool taxonomy
      * 02-05-01. **Financial Research Agent with a Custom Tool: `@function_tool`, hosted and custom tools**
      * 02-05-02. Image Generation and Editing with `ImageGenerationTool`
      * 02-05-03. Multi-Agent Deitel Book Concierge: `FileSearchTool`, RAG, vector stores, handoffs
      * 02-05-04. Code Interpreter Tool: `CodeInterpreterTool`, hosted container
      * 02-05-05. Hosted MCP — Weather and Geocoding: `MCPServerStreamableHttp`, MCP authentication
      * 02-05-06. Local MCP — SQLite Database: `MCPServerStdio`
      * 02-05-07. AccuWeather Agent with `ComputerTool`: `AsyncComputer`, Playwright, `LocalBrowserComputer`
      * 02-05-08. `ShellTool` Folder Inspector: custom executor, human-in-the-loop approval
      * 02-05-09. Local LLM via LiteLLM and Ollama: `LitellmModel`
      * 02-05-10. Python Code Tutor with Dynamic Instructions: callable instructions, `RunContextWrapper`, typed context
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Creating Agents with the OpenAI Agents SDK
---
# Financial Research Agent with a Custom Tool and the Hosted `WebSearchTool`
* `@function_tool` turns any Python function into a model-callable custom tool
* `WebSearchTool` is a predefined OpenAI-hosted tool enabling agents to perform web searches
---

## `yfinance` Module
* Lightweight financial data library
* No key required
* For production financial apps, use a licensed market-data provider appropriate to your use case
* `Ticker` object's `fast_info` gives basic quote fields
    * Stock and ETF prices are live during market hours and otherwise reflect the most recent close
    * Cryptocurrency symbols such as `BTC-USD` and `ETH-USD` trade continuously

---

In [ ]:
import json

import yfinance as yf # Python module for accessing weather data via Yahoo!
from IPython.display import Markdown, display

from agents import (
    Agent, 
    ModelSettings, # for configuring reasoning effort and other model settings
    Runner, 
    function_tool, # decorator that exposes Python functions as tools
    WebSearchTool, # hosted OpenAI tool for web searches
    trace 
)

## Custom Tool: `get_quote`

* `@function_tool` turns a Python function into a tool an agent can call
* SDK uses function's signature and docstring to generate tool schema for the model
* If tool raises an exception, SDK converts it to an error message the agent can describe

---

In [ ]:
@function_tool
def get_quote(symbol: str) -> str:
    """
    Get the current price and basic stats for a stock,  
    mutual fund, ETF or cryptocurrency. 

    Use standard stock/fund/etf ticker symbols: AAPL, MSFT, SPY, QQQ.
    For cryptocurrencies append -USD: BTC-USD, ETH-USD, SOL-USD.

    During market hours, price reflects the live current price.
    Outside market hours, price reflects the last closing price.
    Cryptocurrencies trade 24/7 so price is always live.

    Args:
        symbol: Required ticker symbol, such as AAPL, MSFT, SPY, BTC-USD.
    
    Returns:
        JSON containing the symbol, price, currency, day change, 
        52-week high, 52-week low, market cap and volume.
    """
    info  = yf.Ticker(symbol.upper()).fast_info
    last_price = round(info.last_price or 0, 2)
    previous_close = round(info.previous_close or 0, 2)
    return json.dumps({
        'symbol':     symbol.upper(),
        'price':      last_price,
        'currency':   info.currency,
        'day_change': round(last_price - previous_close, 2),
        '52w_high':   round(info.year_high or 0, 2),
        '52w_low':    round(info.year_low or 0, 2),
        'market_cap': round(info.market_cap or 0, 2),
        'volume':     info.last_volume
    })

## Tool Schema and Direct Testing
* Before giving a custom tool to an agent, inspect and test it
* `params_json_schema` shows the JSON schema the SDK generated from the signature and docstring
* Direct testing catches data or formatting problems before the model is involved

---

In [ ]:
print(f'Tool name: {get_quote.name}')
print()
print('Parameter schema:')
print(json.dumps(get_quote.params_json_schema, indent=2))

In [ ]:
# verify yfinance is returning data
for symbol in ['AAPL', 'MSFT', 'BTC-USD']:
    info  = yf.Ticker(symbol.upper()).fast_info
    price = round(info.last_price or 0, 2)
    prev  = round(info.previous_close or 0, 2)
    print(f'{symbol:8}  {info.currency}  {price:>10.2f}  '
          f'change: {round(price - prev, 2):>+8.2f}')

---

## Multiple Calls to the Same Tool

* A single prompt can require more than one call to the same tool
* Example prompt:
    > Compare AAPL and MSFT.
* The model decides the tool plan
* Parallel tool calls are not guaranteed, but independent quote lookups are a natural fit
* Model may call multiple custom function tools in one turn
    * Not guaranteed
    * Design code to handle one or many tool calls
    * Set `parallel_tool_calls=False` when you run the agent to allow at most one tool call per turn
    * One built-in tool call per turn
    * https://developers.openai.com/api/docs/guides/function-calling
---

## The Financial Research Agent

In [ ]:
FINANCIAL_RESEARCH_INSTRUCTIONS = """
You are a financial research assistant with access to a live quote tool.

Call get_quote exactly once for each ticker symbol to get:
- Current price and key statistics for stocks, ETFs, mutual funds, or 
cryptocurrencies
- Day change, 52-week range, market cap, and volume

For a ticker symbol, always report the asset type and current price. 
Note whether prices are live during market hours or reflect the most recent close.

Use the web_search tool for:
- Recent news, earnings reports, and analyst commentary
- Company announcements, product launches, or regulatory filings
- Broader market context and sector trends

IMPORTANT: Escape ALL symbols in your response that cause Markdown renderers
to think content is a LaTeX formula (for example, $ signs).

Always end your response with this disclaimer:
"This information is for informational purposes only and is not personalized financial
or investment advice. Consult a qualified financial advisor before making investment
decisions."
"""

research_agent = Agent(
    name='FinancialResearchAssistant',
    model='gpt-5.5',
    model_settings=ModelSettings(reasoning={'effort': 'medium'}),
    instructions=FINANCIAL_RESEARCH_INSTRUCTIONS,
    tools=[get_quote, WebSearchTool()]
)

---
## Visualizing the Agent Workflow

In [ ]:
from agents.extensions.visualization import draw_graph
draw_graph(research_agent, filename="agent_workflow.png")

## Compare Two Stocks
* This prompt gives the agent a reason to call `get_quote` for two ticker symbols.

In [ ]:
prompt = """Get the current price, day change, 52-week range, market cap,
and volume for AAPL and MSFT. Compare the two quote snapshots and provide
the latest guidance on whether to buy, hold or sell each."""
print(f'USER: {prompt}')

with trace('02-05-01-financial-research-compare'):
    result = await Runner.run(research_agent, prompt)
    
display(Markdown(result.final_output))

## Documentation References

* [Agents SDK tools](https://openai.github.io/openai-agents-python/tools/)
* [yfinance fast_info reference](https://ranaroussi.github.io/yfinance/reference/yfinance.scrapers.quote.FastInfo.html)

---

© 2026 by Deitel & Associates, Inc. All Rights Reserved.